In [1]:
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download("punkt_tab")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ML\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ML\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# ===============================
# 1. Preprocessing
# ===============================
def preprocess(text):

    tokens = word_tokenize(text.lower())

    lemmatizer = WordNetLemmatizer()

    tokens = [
        lemmatizer.lemmatize(w)
        for w in tokens
        if w.isalpha()
    ]

    return tokens

In [5]:
# ===============================
# 2. Vocabulary
# ===============================
def build_vocab(tokens):

    vocab = list(set(tokens))

    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for i, w in enumerate(vocab)}

    return vocab, word2idx, idx2word


In [7]:
# ===============================
# 3. One Hot Encoding
# ===============================
def one_hot(word, word2idx, size):

    vec = np.zeros(size)

    if word in word2idx:
        vec[word2idx[word]] = 1

    return vec


In [9]:
# ===============================
# 4. Create Skip-Gram Data
# ===============================
def create_data(tokens, window=2):

    X, Y = [], []

    for i, target in enumerate(tokens):

        for j in range(max(0, i - window), min(len(tokens), i + window + 1)):

            if i != j:
                X.append(target)
                Y.append(tokens[j])

    return X, Y

In [11]:
# ===============================
# 5. Softmax
# ===============================
def softmax(x):

    e = np.exp(x - np.max(x))
    return e / np.sum(e)


In [13]:
# ===============================
# 6. Forward Propagation
# ===============================
def forward(x, W1, W2):

    h = np.dot(x, W1)        # Embedding
    y = np.dot(h, W2)

    return h, softmax(y)


In [15]:
# ===============================
# 7. Backward Propagation
# ===============================
def backward(x, h, y, t, W2):

    error = y - t

    dW2 = np.outer(h, error)
    dW1 = np.outer(x, np.dot(W2, error))

    return dW1, dW2

In [17]:
# ===============================
# 8. Train Word2Vec
# ===============================
def train(X, Y, word2idx, size, embed=10, lr=0.01, epochs=500):

    W1 = np.random.rand(size, embed)
    W2 = np.random.rand(embed, size)

    for e in range(epochs):

        loss = 0

        for target, context in zip(X, Y):

            x = one_hot(target, word2idx, size)
            t = one_hot(context, word2idx, size)

            h, y = forward(x, W1, W2)

            loss += -np.sum(t * np.log(y + 1e-9))

            dW1, dW2 = backward(x, h, y, t, W2)

            W1 -= lr * dW1
            W2 -= lr * dW2

        if e % 100 == 0:
            print("Epoch:", e, "Loss:", round(loss, 3))

    return W1, W2

In [19]:

# ===============================
# 9. Word Embeddings
# ===============================
def get_embeddings(W1, vocab):

    return {word: W1[i] for i, word in enumerate(vocab)}


# ===============================
# 10. Similar Words
# ===============================
def cosine(a, b):

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def similar_words(word, embeddings, top=3):

    scores = {}

    for w, vec in embeddings.items():
        if w != word:
            scores[w] = cosine(embeddings[word], vec)

    return sorted(scores, key=scores.get, reverse=True)[:top]


In [27]:
# ===============================
# 11. Main
# ===============================
def main():

    corpus = """
    Flowers are beautiful and colorful.
    Roses are red and smell very nice.
    Sunflowers are yellow and grow tall.
    Plants need water and sunlight to grow.
    Gardeners take care of flowers and plants.
    Bees visit flowers to collect nectar.
    Flowers bloom in spring season.
    Nature looks fresh with blooming flowers.
    """

    tokens = preprocess(corpus)
    print("Tokens:", tokens)

    vocab, word2idx, idx2word = build_vocab(tokens)
    size = len(vocab)

    X, Y = create_data(tokens)

    W1, W2 = train(X, Y, word2idx, size)

    embeddings = get_embeddings(W1, vocab)

    print("\nEmbedding Example:")
    print("flower →", embeddings["flower"])   # singular (important)

    print("\nSimilar to 'flower':")
    print(similar_words("flower", embeddings))


# ===============================
# Run
# ===============================
if __name__ == "__main__":
    main()

Tokens: ['flower', 'are', 'beautiful', 'and', 'colorful', 'rose', 'are', 'red', 'and', 'smell', 'very', 'nice', 'sunflower', 'are', 'yellow', 'and', 'grow', 'tall', 'plant', 'need', 'water', 'and', 'sunlight', 'to', 'grow', 'gardener', 'take', 'care', 'of', 'flower', 'and', 'plant', 'bee', 'visit', 'flower', 'to', 'collect', 'nectar', 'flower', 'bloom', 'in', 'spring', 'season', 'nature', 'look', 'fresh', 'with', 'blooming', 'flower']
Epoch: 0 Loss: 691.143
Epoch: 100 Loss: 413.086
Epoch: 200 Loss: 364.821
Epoch: 300 Loss: 357.144
Epoch: 400 Loss: 355.305

Embedding Example:
flower → [-1.41975073 -0.91646672  1.07036398  0.75122297  1.39066366 -0.36318439
  0.18410693  0.44815095  1.92289165  1.30415996]

Similar to 'flower':
['bee', 'nectar', 'visit']
